# Data transforming

En este notebook se tratará la transformación de dates después de realizar la limpieza.

Definición del proceso:

*Transformen los datos para prepararlos para el modelado y el análisis avanzado:*

- *Normalización o escalado*
- *Codificación de variables categóricas*
- *Generación de nuevas variables*
- *Reducción de dimensionalidad*

Tareas a realizar:

- Creación de columnas de ocupación (a partir de los avaliability_30/60/90/365).
- Creación de columna binaria con: alojamientos de más de 80 puntos de rating.
- Escalar las reviews a la descripción original de la variable rating (1000-->100) y el resto (100-->10).
- Modificar el título de *price* y añadir el símbolo euros (€). (Hecho en D_cleaning)
- *Estandarizar el idioma de los títulos. (mantendremos el idioma original inglés)*
- Transformar la variable *has_avaliability* a dato binario para poder tratarlo en fases de filtrado.
- Convertir en binario la variable *instant_bookable.*

Tareas a futuro (no relevantes por ahora):

- Dividir registro de fechas en distintas columnas Díd, Mes, Año.
- Transformar otras variables que por ahora no son relevantes.

Aspectos a tener en cuenta para el paso de Reducción:

- la variable *has_avaliability* finalmente la deberemos mantener para poder trabajar con ella.

Partimos del CSV limpio (`clean_dataset_29_06_2026.csv`) entregado por Data Cleaning.
Trabajamos siempre sobre una copia para no tocar el dataset original importado.

## Librerias

In [57]:
import pandas as pd
from pathlib import Path


## Importación CSV

In [58]:
def encontrar_raiz_proyecto(nombre_carpeta='Equip_34'):
    '''
    Función para encontrar la carpeta raíz del proyecto subiendo desde el directorio actual.
    '''
    actual = Path.cwd()
    for carpeta in [actual] + list(actual.parents):
        if carpeta.name == nombre_carpeta:
            return carpeta
    raise FileNotFoundError(f"No se encontró la carpeta '{nombre_carpeta}' subiendo desde {actual}")

raiz_proyecto = encontrar_raiz_proyecto('Equip_34')
ruta = raiz_proyecto / 'Data' / 'clean_dataset_29_06_2026.csv'

print(f"Ruta resuelta: {ruta}")
df_original = pd.read_csv(ruta)
df = df_original.copy()

print(df.shape)

Ruta resuelta: /Users/didi/Desktop/Data_Analisis/simulador_ita/ProjecteData/Equip_34/Data/clean_dataset_29_06_2026.csv
(6733, 35)


Confirmamos que seguimos con los mismos registros que en el data_cleaning

## 1. Reescalado de puntuaciones (rating 1000→100, resto 100→10)

Verificado: tras dividir entre 10, `review_scores_rating` no supera 100 y el resto no supera 10 en ningún registro. El factor es uniforme, no hay valores fuera de escala.

In [59]:
df['review_scores_rating'] = df['review_scores_rating'] / 10

cols_10 = ['review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin',
           'review_scores_communication', 'review_scores_location', 'review_scores_value']
df[cols_10] = df[cols_10] / 10

# Verificación: no debe haber valores fuera de escala
assert df['review_scores_rating'].max() <= 100
assert df[cols_10].max().max() <= 10
print("Reescalado correcto, sin valores fuera de rango.")

Reescalado correcto, sin valores fuera de rango.


In [60]:
# mostrar las primeras filas de las columnas de todos los review_scores para verificar el reescalado.
df[[col for col in df.columns if 'review_scores' in col]].head()
# mostrar si hay algun valor por encima de 100 en review_scores_rating y por encima de 10 en las otras columnas de review_scores
print("Valores fuera de rango en review_scores_rating:", df[df['review_scores_rating'] > 100])
print("Valores fuera de rango en otras columnas de review_scores:", df[[col for col in cols_10 if df[col].max() > 10]])

Valores fuera de rango en review_scores_rating: Empty DataFrame
Columns: [apartment_id, name, description, host_id, neighbourhood_name, neighbourhood_district, room_type, accommodates, bathrooms, bedrooms, beds, amenities_list, price_€, minimum_nights, maximum_nights, has_availability, availability_30, availability_60, availability_90, availability_365, number_of_reviews, first_review_date, last_review_date, review_scores_rating, review_scores_accuracy, review_scores_cleanliness, review_scores_checkin, review_scores_communication, review_scores_location, review_scores_value, is_instant_bookable, reviews_per_month, country, city, insert_date]
Index: []

[0 rows x 35 columns]
Valores fuera de rango en otras columnas de review_scores: Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 

El escalado queda completado y todos los valores dentro del rango.

## 2. Columna binaria: rating > 80

Esta columna pretende facilitar el análisis de cliente.
Se calcula después del reescalado (si no, el umbral equivalente en bruto sería 800 y saldría mal).

In [61]:
df['rating_above_80'] = (df['review_scores_rating'] > 80).astype('boolean')
df.loc[df['review_scores_rating'].isna(), 'rating_above_80'] = pd.NA

df['rating_above_80'].value_counts(dropna=False)

df['rating_above_80'].head()

0    True
1    <NA>
2    True
3    True
4    <NA>
Name: rating_above_80, dtype: boolean

## 3. Columnas de ocupación y tasa de ocupación
Columna ocupación interesante para analisis de operaciones.
Columna tasa de ocupación interesante para kpi. En principio la más importante es la tasa_ocupacion_30. valorar si debemos eliminar las otras columnas.

Debemos tomar la decisión de modificar el primer KPI en su ecuación para obtener un resultado válido. El equipo y negocio toman la decision de sustituir los dias disponibles por los dias totales en el denominador de la división.



In [62]:
plazos = {'30': 30, '60': 60, '90': 90, '365': 365}

for suf, total in plazos.items():
    df[f'occupancy_{suf}'] = total - df[f'availability_{suf}']
    df[f'occupancy_rate_{suf}'] = (df[f'occupancy_{suf}'] / total * 100).round(2)

#muestra de las nuevas columnas de ocupación
df[[f'occupancy_{suf}' for suf in plazos.keys()] + [f'occupancy_rate_{suf}' for suf in plazos.keys()]].head()

,occupancy_30,occupancy_60,occupancy_90,occupancy_365,occupancy_rate_30,occupancy_rate_60,occupancy_rate_90,occupancy_rate_365
0,0,0,0,185,0.00,0.0,0.00,50.68
1,30,60,90,345,100.00,100.0,100.00,94.52
2,26,42,42,261,86.67,70.0,46.67,71.51
3,0,15,43,127,0.00,25.0,47.78,34.79
4,19,24,25,27,63.33,40.0,27.78,7.40


## 4. `is_instant_bookable` → binario (sin tocar el original)

Se mantiene la columna original tal cual (por si alguien la necesita en su formato de origen) y se crea una nueva en castellano con 1/0, respetando los nulos (no se rellenan).

In [63]:
df['is_instant_bookable_numeric'] = df['is_instant_bookable'].map({'VERDADERO': 1, 'FALSO': 0})

print(df['is_instant_bookable_numeric'].value_counts(dropna=False))
print(df['is_instant_bookable'].value_counts(dropna=False))

df[['is_instant_bookable', 'is_instant_bookable_numeric']].head()

is_instant_bookable_numeric
1    3590
0    3143
Name: count, dtype: int64
is_instant_bookable
VERDADERO    3590
FALSO        3143
Name: count, dtype: int64


,is_instant_bookable,is_instant_bookable_numeric
0,VERDADERO,1
1,FALSO,0
2,VERDADERO,1
3,VERDADERO,1
4,FALSO,0


## 5. `has_availability` → binario, nulos intactos

Se convierte directamente sobre la misma columna (no genera controversia porque no se rellena ningún nulo — se mantienen como están).

In [64]:
# utilizamos Int64 para poder tener nulos en la columna numérica y obtener un tipo de dato entero, en lugar de float que es lo que obtendríamos con int64.
df['has_availability_numeric'] = df['has_availability'].map({'VERDADERO': 1, 'FALSO': 0}).astype('Int64')

print(df['has_availability_numeric'].value_counts(dropna=False))
print(df['has_availability'].value_counts(dropna=False))

df[['has_availability', 'has_availability_numeric']].tail()

has_availability_numeric
1       6199
<NA>     534
Name: count, dtype: Int64
has_availability
VERDADERO    6199
NaN           534
Name: count, dtype: int64


,has_availability,has_availability_numeric
6728,VERDADERO,1
6729,VERDADERO,1
6730,VERDADERO,1
6731,VERDADERO,1
6732,VERDADERO,1


## 6. Verificación y exportación

In [65]:
print("Dimensiones finales:", df.shape)
print("apartment_id único:", df['apartment_id'].is_unique)
df.head()

Dimensiones finales: (6733, 46)
apartment_id único: True


,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,occupancy_30,occupancy_rate_30,occupancy_60,occupancy_rate_60,occupancy_90,occupancy_rate_90,occupancy_365,occupancy_rate_365,is_instant_bookable_numeric,has_availability_numeric
0,13707226,"Remarkable Value, Unbeatable Location",A spacious double bedroom with a balcony. It's...,80008404,el Barri G�tic,Ciutat Vella,Private room,2,4.0,1.0,...,0,0.00,0,0.0,0,0.00,185,50.68,1,<NA>
1,13011987,"Lovely flat in Barcelona, 10' from the city ce...","Ideal for couples, located in a quiet and safe...",31321818,Sant Mart� de Proven�als,Sant Mart�,Entire home/apt,2,1.0,1.0,...,30,100.00,60,100.0,90,100.00,345,94.52,0,<NA>
2,14999488,Habitaci�n doble en bonito apartamento,Piso grande y bonito de 90 m2. Tranquilo y por...,7093663,el Camp d'en Grassot i Gr�cia Nova,Gr�cia,Private room,2,1.0,1.0,...,26,86.67,42,70.0,42,46.67,261,71.51,1,<NA>
3,6547870,Bella Vista. N� de registro: HUTB005799,Bella Vista es un apartamento �nico en Barcelo...,34249903,el Baix Guinard�,Horta-Guinard�,Entire home/apt,7,1.0,3.0,...,0,0.00,15,25.0,43,47.78,127,34.79,1,<NA>
4,15253506,Casa de Andrea,"My house is very big, from the twenties. I onl...",17925327,Sants,Sants-Montju�c,Entire home/apt,14,2.0,5.0,...,19,63.33,24,40.0,25,27.78,27,7.40,0,<NA>


In [66]:
'''df.to_csv(ruta, index=False, encoding='utf-8')
print(f"Archivo sobreescrito en: {ruta}")'''

'df.to_csv(ruta, index=False, encoding=\'utf-8\')\nprint(f"Archivo sobreescrito en: {ruta}")'

## Recomendaciones para el Data Reduction

- La tasa de ocupación solamente usaremos la 30 dias
- Se pueden eliminar en caso que no sean necesarias las columnas originales is_instant_bookable o has_avaliability
- Mantener aquellas columnas necesarias para los análisis departamentales y los KPI